# Allan - Обучение собственной нейросети с нуля

Этот notebook для обучения русскоязычной языковой модели Allan с нуля на Google Colab.

## Возможности:
- Создание модели с нуля (не дообучение!)
- Загрузка датасетов с Google Drive или corus
- Обучение токенизатора на русском языке
- Сохранение чекпоинтов на Google Drive
- Возобновление обучения
- Сохранение финальной модели

## 1. Настройка окружения

In [ ]:
# Проверка GPU
!nvidia-smi

In [ ]:
# Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Устанавливаем зависимости
!pip install -q torch transformers accelerate datasets sentencepiece corus razdel pymorphy2 tqdm pyyaml

print("✓ Зависимости установлены")

In [ ]:
# Клонируем проект Allan (если ещё не склонирован)
import os

if not os.path.exists('/content/Allan'):
    !git clone https://github.com/KEYTRON/Allan.git
    %cd Allan
else:
    %cd /content/Allan
    !git pull

# Добавляем src в путь
import sys
sys.path.insert(0, '/content/Allan/src')

## 2. Конфигурация

In [ ]:
# ========== КОНФИГУРАЦИЯ ==========

# Пути на Google Drive
DRIVE_BASE = "/content/drive/MyDrive/Allan_Model"
DATASET_PATH = f"{DRIVE_BASE}/datasets"  # Где лежат датасеты
CHECKPOINTS_PATH = f"{DRIVE_BASE}/checkpoints"  # Сохранение чекпоинтов
MODELS_PATH = f"{DRIVE_BASE}/models"  # Финальные модели
TOKENIZER_PATH = f"{DRIVE_BASE}/tokenizer"  # Обученный токенизатор

# Создаем директории
for path in [DATASET_PATH, CHECKPOINTS_PATH, MODELS_PATH, TOKENIZER_PATH]:
    os.makedirs(path, exist_ok=True)

# Параметры модели
MODEL_CONFIG = {
    'vocab_size': 50000,
    'n_embd': 512,        # Размерность эмбеддингов (можно увеличить для большей модели)
    'n_layer': 8,         # Количество слоев (можно увеличить)
    'n_head': 8,          # Количество голов внимания
    'n_positions': 1024,  # Максимальная длина контекста
    'dropout': 0.1,
}

# Параметры обучения
TRAINING_CONFIG = {
    'batch_size': 16,      # Размер батча (уменьшить если не хватает памяти)
    'max_epochs': 10,
    'learning_rate': 3e-4,
    'warmup_steps': 1000,
    'max_seq_length': 512,
    'gradient_accumulation_steps': 4,  # Для эффективности при малом batch_size
    'save_every': 1000,    # Сохранять каждые N шагов
}

# Источник данных
DATA_SOURCE = 'drive'  # 'drive' или 'corus'

# Если 'drive', укажите путь к файлу
DRIVE_DATASET_FILE = "Colab Notebooks/dataset.jsonl"  # Относительно MyDrive

print("✓ Конфигурация загружена")
print(f"Модель: {MODEL_CONFIG['n_layer']} слоев, {MODEL_CONFIG['n_embd']} размерность")
print(f"Обучение: batch={TRAINING_CONFIG['batch_size']}, epochs={TRAINING_CONFIG['max_epochs']}")

## 3. Подготовка данных

In [ ]:
# Импортируем модули Allan
from model.tokenizer import AllanTokenizer
from datasets.data_loader import CorusDataLoader, GoogleDriveDataLoader, prepare_training_data

print("✓ Модули импортированы")

In [ ]:
# Собираем тексты для обучения токенизатора
print("Сбор текстов для обучения токенизатора...")

texts_for_tokenizer = []

if DATA_SOURCE == 'corus':
    loader = CorusDataLoader()
    for i, text in enumerate(loader.load_all_available(max_texts=100000)):
        texts_for_tokenizer.append(text)
        if (i + 1) % 10000 == 0:
            print(f"Собрано текстов: {i + 1}")
        if i >= 100000:
            break
else:
    # Загружаем с Drive
    drive_loader = GoogleDriveDataLoader(drive_path="/content/drive/MyDrive")
    
    import json
    for i, record in enumerate(drive_loader.load_jsonl(DRIVE_DATASET_FILE)):
        if 'text' in record:
            texts_for_tokenizer.append(record['text'])
        elif 'output' in record:
            texts_for_tokenizer.append(record['output'])
        
        if (i + 1) % 1000 == 0:
            print(f"Загружено записей: {i + 1}")

print(f"\n✓ Собрано текстов: {len(texts_for_tokenizer)}")

In [ ]:
# Обучаем токенизатор
print("Обучение токенизатора...")

tokenizer = AllanTokenizer(vocab_size=MODEL_CONFIG['vocab_size'])

# Сохраняем тексты во временный файл
temp_file = "/content/tokenizer_train_texts.txt"
with open(temp_file, 'w', encoding='utf-8') as f:
    for text in texts_for_tokenizer[:50000]:  # Используем часть для обучения
        f.write(text + '\n')

# Обучаем
tokenizer.train(
    texts=temp_file,
    model_prefix="/content/allan_tokenizer",
    vocab_size=MODEL_CONFIG['vocab_size']
)

# Сохраняем на Drive
import shutil
shutil.copy("/content/allan_tokenizer.model", f"{TOKENIZER_PATH}/tokenizer.model")
shutil.copy("/content/allan_tokenizer.vocab", f"{TOKENIZER_PATH}/tokenizer.vocab")

tokenizer.save_pretrained(TOKENIZER_PATH)

print("✓ Токенизатор обучен и сохранен")

# Тестируем
test_text = "Привет! Как дела?"
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)
print(f"\nТест: '{test_text}' -> {tokens} -> '{decoded}'")

In [ ]:
# Подготавливаем обучающие данные
print("Подготовка обучающих данных...")

train_data_path = f"{DATASET_PATH}/train_data.pt"

if not os.path.exists(train_data_path):
    if DATA_SOURCE == 'corus':
        data_source = 'corus'
    else:
        data_source = f"/content/drive/MyDrive/{DRIVE_DATASET_FILE}"
    
    train_tensor = prepare_training_data(
        data_source=data_source,
        tokenizer=tokenizer,
        save_path=train_data_path,
        max_samples=None,  # Все данные
        max_length=TRAINING_CONFIG['max_seq_length']
    )
else:
    import torch
    train_tensor = torch.load(train_data_path)
    print(f"✓ Загружены обучающие данные: {train_tensor.shape}")

## 4. Создание модели

In [ ]:
import torch
from model.architecture import AllanGPT, AllanConfig

# Создаем конфигурацию
config = AllanConfig(**MODEL_CONFIG)

# Создаем модель
model = AllanGPT(config)

# Переносим на GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

print(f"✓ Модель создана и перенесена на {device}")
print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 5. Обучение

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import math

class TextDataset(Dataset):
    def __init__(self, data_tensor):
        self.data = data_tensor
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# Создаем датасет и dataloader
dataset = TextDataset(train_tensor)
dataloader = DataLoader(
    dataset,
    batch_size=TRAINING_CONFIG['batch_size'],
    shuffle=True,
    num_workers=2
)

# Оптимизатор
optimizer = AdamW(
    model.parameters(),
    lr=TRAINING_CONFIG['learning_rate'],
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# Scheduler
total_steps = len(dataloader) * TRAINING_CONFIG['max_epochs']
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"✓ Подготовка к обучению завершена")
print(f"Батчей на эпоху: {len(dataloader)}")
print(f"Всего шагов: {total_steps}")

In [ ]:
# Функция обучения
def train_epoch(model, dataloader, optimizer, scheduler, device, grad_accum_steps=1):
    model.train()
    total_loss = 0
    
    optimizer.zero_grad()
    
    pbar = tqdm(dataloader, desc="Training")
    for i, batch in enumerate(pbar):
        batch = batch.to(device)
        
        # Forward
        logits, loss = model(batch, labels=batch)
        loss = loss / grad_accum_steps
        
        # Backward
        loss.backward()
        
        if (i + 1) % grad_accum_steps == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * grad_accum_steps
        pbar.set_postfix({'loss': f'{loss.item() * grad_accum_steps:.4f}'})
    
    return total_loss / len(dataloader)

# Функция сохранения чекпоинта
def save_checkpoint(model, optimizer, scheduler, epoch, step, loss):
    checkpoint_path = f"{CHECKPOINTS_PATH}/checkpoint_epoch{epoch}_step{step}.pt"
    
    torch.save({
        'epoch': epoch,
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
        'config': MODEL_CONFIG,
    }, checkpoint_path)
    
    print(f"✓ Чекпоинт сохранен: {checkpoint_path}")

print("✓ Функции обучения готовы")

In [ ]:
# ОБУЧЕНИЕ
print("="*50)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*50)

global_step = 0
best_loss = float('inf')

for epoch in range(TRAINING_CONFIG['max_epochs']):
    print(f"\n{'='*50}")
    print(f"Эпоха {epoch + 1}/{TRAINING_CONFIG['max_epochs']}")
    print(f"{'='*50}")
    
    # Обучаем эпоху
    avg_loss = train_epoch(
        model,
        dataloader,
        optimizer,
        scheduler,
        device,
        grad_accum_steps=TRAINING_CONFIG['gradient_accumulation_steps']
    )
    
    print(f"\nСредний loss: {avg_loss:.4f}")
    print(f"Perplexity: {math.exp(avg_loss):.2f}")
    
    # Сохраняем чекпоинт
    save_checkpoint(model, optimizer, scheduler, epoch, global_step, avg_loss)
    
    # Сохраняем лучшую модель
    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained(f"{MODELS_PATH}/best_model")
        print(f"✓ Лучшая модель сохранена (loss={best_loss:.4f})")
    
    global_step += len(dataloader)

print("\n" + "="*50)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*50)
print(f"Лучший loss: {best_loss:.4f}")

## 6. Сохранение финальной модели

In [ ]:
# Сохраняем финальную модель
final_model_path = f"{MODELS_PATH}/allan_final"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✓ Финальная модель сохранена в {final_model_path}")
print(f"✓ Токенизатор сохранен в {final_model_path}")

## 7. Тестирование модели

In [ ]:
# Тестируем генерацию
model.eval()

test_prompt = "Россия - это"
print(f"Промпт: {test_prompt}")

# Кодируем
input_ids = torch.tensor([tokenizer.encode(test_prompt)]).to(device)

# Генерируем
with torch.no_grad():
    generated = model.generate(
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )

# Декодируем
generated_text = tokenizer.decode(generated[0].tolist())
print(f"\nСгенерировано:\n{generated_text}")

## 8. Возобновление обучения из чекпоинта (опционально)

In [ ]:
# Если нужно продолжить обучение
def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None):
    checkpoint = torch.load(checkpoint_path)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    if scheduler:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    print(f"✓ Чекпоинт загружен: epoch={checkpoint['epoch']}, loss={checkpoint['loss']:.4f}")
    
    return checkpoint['epoch'], checkpoint['step']

# Пример использования:
# epoch, step = load_checkpoint(f"{CHECKPOINTS_PATH}/checkpoint_epoch0_step1000.pt", model, optimizer, scheduler)